# 第 04 章　空间验证：随机划分的评分为什么可能太乐观？

> **本章问题**：随机抽像元得到的高分，能代表模型到了新地方也准确吗？

先想一个很直观的情况：两个相邻像元的地表温度通常很像。如果我们随机把它们分到训练集和验证集，模型验证时可能就在“邻居”那里见过几乎相同的情况。这样的分数看起来不错，却不一定代表模型能去一个没见过的区域预测。

这一章用同一个 Ridge 模型做对照。我们既看随机划分，也看空间分块，还会固定一块独立测试区，最后比较地图和误差。


## 4.1　先看已经固定好的空间划分

项目已经保存了划分文件 `data/processed/training_table_splits.csv`，本章直接读取，不重新切分。规则是先留出中南部连续区域作独立测试区，再把剩下的训练像元按空间块分成 5 折。

下面先核对每一块有多少像元，以及训练区和测试区的目标均值。请留意测试区和训练区是否真的像“同一批数据的随机两半”。


In [ ]:
import os
import sys
import time
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
T0 = time.perf_counter()
print("Python:", sys.executable)
print("Conda :", os.environ.get("CONDA_PREFIX"))

bundle = gd.load_stack()
bound = gpd.read_file(gd.RAW / "GanNan.shp")
splits = gd.load_splits()
splits.to_csv(gd.TABLES / "04_splits.csv", index=False, encoding="utf-8-sig")

split_view = gd.split_summary(splits).copy()
split_view["占比"] = (split_view["占比"] * 100).round(1).astype(str) + "%"
display(split_view)

train = splits[splits["split"] == "train"].reset_index(drop=True)
test = splits[splits["split"] == "test"].reset_index(drop=True)
print("训练区像元:", len(train), "| 独立测试区像元:", len(test),
      f"（{len(test) / len(splits) * 100:.1f}%）")
print("测试区 row:", int(test.row.min()), "–", int(test.row.max()),
      "| col:", int(test.col.min()), "–", int(test.col.max()))
print("空间折像元数:", train.fold.value_counts().sort_index().to_dict())
print(f"训练区 LST 均值/标准差: {train.lst.mean():.2f} / {train.lst.std():.2f}")
print(f"测试区 LST 均值/标准差: {test.lst.mean():.2f} / {test.lst.std():.2f}（仅描述，不评分）")

测试区 1360 个像元，占全表 19.0%，是一个 36 行 × 41 列的连续矩形（有效像元略少于矩形面积）。5 个空间折分别为 1100、1215、1010、1418、1053 个像元，大小不完全相等，但同折像元在地图上成块。训练区均值 11.95，测试区均值 12.83，测试区整体偏暖约 0.88（原始像元值）——这说明留出区不是训练区的“随机缩影”，外推本来就会更难。

## 4.2 划分长什么样

把 `fold` 放回 176×106 网格：五种颜色是 5 个空间折，浅灰是独立测试区，黑色虚线框出中南测试块。

In [ ]:
gs.plot_split_map(
    splits, bundle["shape"], bundle["extent"], "04_split_map.png",
    boundary=bound, crs=bundle["crs"],
    title="固定空间划分：5 个训练折 + 中南连续测试区（1360 像元）",
)
print("已保存 04_split_map.png")

## 4.3　随机 5 折与空间 5 折对比

地图上能看到，五折是成片分配的，独立测试区也完整留在中南部，没有被切碎撒进训练区。这样的验证更接近一个实际问题：如果我们把一片区域留起来，模型到了那里还能不能用？

空间折的块边界会受块大小和随机种子影响。为了让后面各章能公平比较，我们沿用这份固定划分，不在这里反复调整。


In [ ]:
X, y = gm.get_xy(train)
rows_arr = train["row"].to_numpy()
cols_arr = train["col"].to_numpy()


def neighbor_fraction(val_idx, fit_idx):
    '''验证像元中，四邻域至少有一个训练像元的比例。'''
    fit = set(zip(rows_arr[fit_idx], cols_arr[fit_idx]))
    hit = 0
    for i in val_idx:
        r, c = rows_arr[i], cols_arr[i]
        if ((r + 1, c) in fit or (r - 1, c) in fit
                or (r, c + 1) in fit or (r, c - 1) in fit):
            hit += 1
    return hit / len(val_idx)


fold_rows = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (fit_idx, val_idx) in enumerate(kf.split(X), start=1):
    model = gm.ridge_model()
    model.fit(X.iloc[fit_idx], y[fit_idx])
    pred = model.predict(X.iloc[val_idx])
    fold_rows.append({
        "方案": "随机 5 折", "折": i, "验证像元数": len(val_idx),
        **gm.metrics(y[val_idx], pred),
        "邻接训练像元比例": neighbor_fraction(val_idx, fit_idx),
    })

for f in range(gm.N_FOLDS):
    val_idx = np.where(train["fold"].to_numpy() == f)[0]
    fit_idx = np.where(train["fold"].to_numpy() != f)[0]
    model = gm.ridge_model()
    model.fit(X.iloc[fit_idx], y[fit_idx])
    pred = model.predict(X.iloc[val_idx])
    fold_rows.append({
        "方案": "空间 5 折", "折": f + 1, "验证像元数": len(val_idx),
        **gm.metrics(y[val_idx], pred),
        "邻接训练像元比例": neighbor_fraction(val_idx, fit_idx),
    })

fold_scores = pd.DataFrame(fold_rows)
fold_scores.to_csv(gd.TABLES / "04_fold_scores.csv", index=False, encoding="utf-8-sig")

comp_rows = []
for name, part in fold_scores.groupby("方案", sort=False):
    comp_rows.append({
        "方案": name,
        "折数": len(part),
        "验证像元合计": int(part["验证像元数"].sum()),
        "MAE 均值": part["MAE"].mean(),
        "MAE 标准差": part["MAE"].std(ddof=1),
        "RMSE 均值": part["RMSE"].mean(),
        "RMSE 标准差": part["RMSE"].std(ddof=1),
        "R2 均值": part["R2"].mean(),
        "R2 标准差": part["R2"].std(ddof=1),
        "邻接训练像元比例均值": part["邻接训练像元比例"].mean(),
    })
comp = pd.DataFrame(comp_rows)
comp.to_csv(gd.TABLES / "04_random_vs_spatial.csv", index=False, encoding="utf-8-sig")
display(comp.round(3))

by_name = comp.set_index("方案")
gap_rmse = by_name.loc["随机 5 折", "RMSE 均值"] - by_name.loc["空间 5 折", "RMSE 均值"]
gap_r2 = by_name.loc["随机 5 折", "R2 均值"] - by_name.loc["空间 5 折", "R2 均值"]
print(f"随机 − 空间：RMSE {gap_rmse:+.4f}，R² {gap_r2:+.4f}")
print(f"RMSE 相对低 {abs(gap_rmse) / by_name.loc['空间 5 折', 'RMSE 均值'] * 100:.1f}%；"
      f"R² 相对高 {gap_r2 / by_name.loc['空间 5 折', 'R2 均值'] * 100:.1f}%")

把结果画在同一张图上：每个点是一折，横线是该方案的折均。两个面板各自使用同一纵轴尺度，左图越低越好，右图越高越好。

In [ ]:
schemes = ["随机 5 折", "空间 5 折"]
colors = {"随机 5 折": "#d97706", "空间 5 折": "#2a6f97"}
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.6), constrained_layout=True)
panels = [
    (axes[0], "RMSE", "RMSE（原始像元值）", "(a) RMSE：越低越好"),
    (axes[1], "R2", "R²", "(b) R²：越高越好"),
]
for ax, metric, ylabel, title in panels:
    for i, name in enumerate(schemes):
        vals = fold_scores.loc[fold_scores["方案"] == name, metric].to_numpy()
        x = i + rng.uniform(-0.085, 0.085, size=len(vals))
        ax.scatter(x, vals, s=48, color=colors[name], edgecolor="white",
                   linewidth=0.7, zorder=3, label=name)
        mean = vals.mean()
        ax.hlines(mean, i - 0.25, i + 0.25, color=colors[name], lw=2.2, zorder=4)
        ax.annotate(f"均值 {mean:.3f}", xy=(i + 0.29, mean), ha="left", va="center",
                    fontsize=9, color=colors[name], fontweight="semibold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(schemes)
    ax.set_xlim(-0.45, 1.55)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", color=gs.GRID, lw=0.6, alpha=0.9)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

axes[0].set_ylim(1.10, 1.72)
axes[1].set_ylim(-0.02, 0.56)
axes[1].axhline(0, color="#888888", lw=0.8, ls=":")
axes[0].legend(loc="upper left", fontsize=8.8)
fig.suptitle("同一 Ridge 模型、同一训练区 5796 行：随机 5 折 vs 空间 5 折",
             fontsize=13.5, fontweight="semibold")
fig.text(
    0.5, -0.035,
    f"折均差异：RMSE {gap_rmse:+.3f}（随机更低 {abs(gap_rmse) / by_name.loc['空间 5 折', 'RMSE 均值'] * 100:.1f}%），"
    f"R² {gap_r2:+.3f}（随机更高 {gap_r2 / by_name.loc['空间 5 折', 'R2 均值'] * 100:.1f}%）",
    ha="center", fontsize=9, color=gs.INK,
)
gs.save(fig, "04_random_vs_spatial.png")
plt.close(fig)
print("已保存 04_random_vs_spatial.png")

结果很清楚：随机 5 折的 RMSE 折均是 1.243，空间 5 折是 1.339，随机划分把误差压低了约 0.096（7.2%）；R² 从 0.304 抬到 0.470，高出约 0.166（相对 55%）。更值得注意的是折间差异：随机折的 RMSE 在 1.203–1.279 之间、R² 在 0.443–0.517 之间，非常整齐；空间折的 RMSE 从 1.158 到 1.656，R² 从 0.059 到 0.431，最差的第 3 折 R² 只有 0.06。折均 0.304 掩盖了“有的片区几乎预测不了”的事实。

**随机评分更好，不代表模型更强。** 它只是把“相邻像元”放进了训练集。空间折更接近“预测一个没观测过的片区”，所以分数更低、波动更大——这才是我们做留出区域估计时该预期的难度。

## 4.4 乐观偏差从哪来：邻接与空间自相关

如果 LST 在空间上独立，随机划分和空间划分应该给出相近的分数。实际并非如此。下面算两个量：LST 与东西/南北相邻像元的相关（滞后 1 格），以及两种划分下“验证像元有四邻域训练点”的比例。

In [ ]:
valid = bundle["valid"]
lst_grid = bundle["target"]


def lag1_corr(axis, shift):
    shifted = np.roll(lst_grid, shift, axis=axis)
    mask = valid & np.roll(valid, shift, axis=axis)
    return float(np.corrcoef(lst_grid[mask], shifted[mask])[0, 1]), int(mask.sum())


corr_ew, n_ew = lag1_corr(axis=1, shift=1)
corr_ns, n_ns = lag1_corr(axis=0, shift=1)
print(f"LST 东西向滞后 1 格相关: {corr_ew:.3f}（{n_ew} 对像元）")
print(f"LST 南北向滞后 1 格相关: {corr_ns:.3f}（{n_ns} 对像元）")

neighbor_summary = (fold_scores
                    .groupby("方案", sort=False)["邻接训练像元比例"]
                    .agg(["mean", "min", "max"]).round(3))
display(neighbor_summary)
for name in ["随机 5 折", "空间 5 折"]:
    vals = fold_scores.loc[fold_scores["方案"] == name, "邻接训练像元比例"]
    print(f"{name}：验证像元邻接训练点的比例 {vals.min():.1%}–{vals.max():.1%}，"
          f"折均 {vals.mean():.1%}")

LST 与东西邻像元的相关高达 0.944，与南北邻像元也有 0.892。随机 5 折中，99.3%–99.9% 的验证像元至少有一个四邻域像元在训练集里；空间 5 折只有 10.5%–17.4%，折均 13.8%。差距接近 86 个百分点。

这就是机制：随机验证点几乎总能从紧邻的训练点“抄答案”，而 Ridge 虽然简单，也能借助 DEM/TEM 等平滑变量把局部水平拟合得很稳；空间折把整块区域拿掉后，验证点周围没有训练点，模型必须真正外推到不同的地形/温度组合，误差自然上升。空间自相关越强、块越小，随机划分的乐观偏差越大。本例的偏差是 RMSE 0.10、R² 0.17；如果换成能记忆局部模式的树模型，差距可能更大（第 06 章会继续比较）。

## 4.5 本章不碰测试区

本章的所有拟合、比较、诊断都只使用训练区 5796 行。中南测试区只被“固定”和“画出”，没有被预测、没有被评分，也没有参与块大小或模型选择。它的作用是等到模型定稿后，提供一次性的、独立的空间外推检验（第 05 章起）。如果现在就拿它调参，它就不再独立。

## 地理视角

甘南的 LST 不是随机变量：它随高程、气温和地表覆盖连续变化，形成片状高低值。随机划分假设样本可交换，等于假设“地图上任意两点可以互换”，这与地理数据的性质冲突。空间划分则承认“位置相邻的观测相似”，用整块留出模拟真实制图的外推场景。测试区在中南、均值比训练区高 0.88，也提醒我们：留出区与训练区的分布并不完全相同，最终测试分数低于空间 CV 并非意外。

## 常见坑

1. **把随机 CV 当最终精度**：随机折的 0.470 不是模型泛化能力，而是“邻域插值能力”。报告精度时必须写清划分方式和块尺寸。
2. **测试区反复使用**：看一眼测试分数、回去改特征、再看一眼，测试区就泄漏了。测试区只能在模型定稿后评一次。
3. **只看折均、不看折间**：空间折 R² 从 0.059 到 0.431，均值 0.304 会掩盖弱区。应同时报告折间范围和最差折。
4. **块尺寸随手定**：块越小，空间折越接近随机折，乐观偏差越大；块越大，折内样本越少、方差越大。块尺寸、种子、折数都要写进方法。
5. **用测试区目标值做描述后继续调参**：本章用测试均值说明分布差异，仅限描述；一旦用它选模型，测试区就不再独立。

## 练习

用 `gd.add_splits(table, block_rows=3, block_cols=3)` 在内存中生成更细的 3×3 分块（不要覆盖 `training_table_splits.csv`），保持同样的 5 折和随机种子，重跑本章的 Ridge 空间 CV。比较 RMSE、R² 与本章 6×6 分块的结果，并解释：为什么块越细，分数越接近随机 5 折？

## 小结

- 固定划分 = 中南连续测试区 1360 像元（19.0%）+ 训练区 5796 像元上的 5 个空间折；测试区不参与训练和调参。
- 同一 Ridge 模型：随机 5 折 RMSE 1.243、R² 0.470；空间 5 折 RMSE 1.339、R² 0.304，随机划分明显更乐观。
- 空间折折间波动大（R² 0.059–0.431），折均会掩盖弱区；报告时要给折间范围。
- 机制是空间自相关：LST 邻接相关 0.94/0.89，随机折 99% 以上的验证像元紧邻训练点，空间折只有约 14%。
- 04 只建立验证框架，不评估测试区；最终测试区检验留到模型定稿之后。

In [ ]:
elapsed = time.perf_counter() - T0
fig_names = ["04_split_map.png", "04_random_vs_spatial.png"]
run_summary = {
    "notebook": "04_空间验证.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "运行时间_秒": round(elapsed, 1),
    "关键指标": {
        "训练区像元": int(len(train)),
        "测试区像元": int(len(test)),
        "测试区占比": round(len(test) / len(splits), 4),
        "空间折像元数": {int(k): int(v) for k, v in train.fold.value_counts().sort_index().items()},
        "随机5折_RMSE均值": round(float(by_name.loc["随机 5 折", "RMSE 均值"]), 4),
        "空间5折_RMSE均值": round(float(by_name.loc["空间 5 折", "RMSE 均值"]), 4),
        "RMSE差_随机减空间": round(float(gap_rmse), 4),
        "随机5折_R2均值": round(float(by_name.loc["随机 5 折", "R2 均值"]), 4),
        "空间5折_R2均值": round(float(by_name.loc["空间 5 折", "R2 均值"]), 4),
        "R2差_随机减空间": round(float(gap_r2), 4),
        "LST邻接相关_东西": round(corr_ew, 3),
        "LST邻接相关_南北": round(corr_ns, 3),
        "邻接训练比例_随机": round(float(neighbor_summary.loc["随机 5 折", "mean"]), 3),
        "邻接训练比例_空间": round(float(neighbor_summary.loc["空间 5 折", "mean"]), 3),
        "测试区LST均值_仅描述": round(float(test.lst.mean()), 3),
        "训练区LST均值": round(float(train.lst.mean()), 3),
        "测试区用途": "仅固定与描述，不参与训练、调参和本章评分",
    },
    "图件": fig_names,
    "未核实事项": [
        "LST/TEM 单位与时间聚合方式未注明，全文只写“原始像元值”",
        "降水/土壤水分为 2020 年，与 2021 年 LST 不同期",
        "空间折块尺寸与种子固定为 6×6、42，块尺寸敏感性未在本章展开",
        "案例为方法演示，不构成甘南县 LST 产品的精度结论",
    ],
}
gd.write_json(run_summary, "04_run_summary.json")
print(f"本章执行耗时 {elapsed:.1f} 秒")
print("已保存:", ", ".join(fig_names), "04_run_summary.json")